# HS4002 Week 8

## Practising with Count Models

Research question: **what factors influence the number of children someone has?**

We'll compare three model types for count outcomes:
1. OLS (baseline)
2. Poisson regression
3. Negative Binomial regression

Plus: overdispersion testing and marginal effects.

In [1]:
# !pip install pandas numpy plotnine statsmodels stargazer marginaleffects scipy

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
import statsmodels.formula.api as smf
from scipy.stats import chi2

In [2]:
raw_df = pd.read_csv("gss2022_mini2.csv")

chosen_variables = [
	'id', 'age', 'happy', 'yrlvmus', 'yrartxbt', 'yrmovie', 'yrcreat',
	'income16', 'prestg10', 'degree', 'race', 'sex', 'wrkstat', 'biokids', 'educ'
]
df = raw_df[chosen_variables].dropna().copy()

income_map = {
	1: 500,    2: 2000,   3: 3500,   4: 4500,   5: 5500,   6: 6500,
	7: 7500,   8: 9000,   9: 11250,  10: 13750, 11: 16250, 12: 18750,
	13: 21250, 14: 23750, 15: 27500, 16: 32500, 17: 37500, 18: 45000,
	19: 55000, 20: 67500, 21: 82500, 22: 100000, 23: 120000, 24: 140000,
	25: 160000, 26: 250000
}
df['income_cont']   = df['income16'].map(income_map)
df['income_10k']   = df['income_cont'] / 10000
df['age']       = raw_df.loc[df.index, 'age'] - 18

df['binary_lvmus']  = (df['yrlvmus']  == 1).astype(int)
df['binary_artxbt'] = (df['yrartxbt'] == 1).astype(int)
df['binary_movie']  = (df['yrmovie']  == 1).astype(int)
df['binary_creat']  = (df['yrcreat']  == 1).astype(int)
df['omni'] = df[['binary_lvmus','binary_artxbt','binary_movie','binary_creat']].sum(axis=1)

df['race_bin']   = pd.Categorical(df['race'].astype(str))
df['race_cat']   = df['race'].map({1: 'White', 2: 'Black'}).fillna('Other')
df['ba_binary']  = (df['degree'] >= 3).astype(int)
df['sex_woman']  = pd.Categorical(
    np.where(df['sex'] == 2, 'Woman', 'Not Woman'),
    categories=['Not Woman', 'Woman']
)
df['working']    = np.where(df['wrkstat'] <= 3, 'Working', 'Not Working')
df['happy_bin']  = (df['happy'] <= 2).astype(int)
df['happy_cat']  = np.where(df['happy'] <= 2, 'Happy', 'Not Happy')

# Starting with an OLS model

Regress no. of children (`biokids`) against income in ten thousands (`income_10k`). Note: one unit = $10,000. Name this `ols_lm1`.

In [3]:
ols_lm1 = smf.ols('biokids ~ income_10k', data=df).fit()
print(ols_lm1.summary())

                            OLS Regression Results                            
Dep. Variable:                biokids   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.002
Method:                 Least Squares   F-statistic:                   0.04228
Date:                Fri, 16 Oct 2026   Prob (F-statistic):              0.837
Time:                        13:28:26   Log-Likelihood:                -1050.5
No. Observations:                 557   AIC:                             2105.
Df Residuals:                     555   BIC:                             2114.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.5647      0.107     14.639      0.0

How would you interpret the coefficient for `income_10k`? What is unusual about using OLS on a count variable?

# Continuing with a Poisson Model

Use `smf.poisson()` — equivalent to `glm(..., family='poisson')` in R.

In [4]:
po_lm1 = smf.poisson('biokids ~ income_10k', data=df).fit()
print(po_lm1.summary())

Optimization terminated successfully.
         Current function value: 1.767601
         Iterations 3
                          Poisson Regression Results                          
Dep. Variable:                biokids   No. Observations:                  557
Model:                        Poisson   Df Residuals:                      555
Method:                           MLE   Df Model:                            1
Date:                Fri, 16 Oct 2026   Pseudo R-squ.:               3.454e-05
Time:                        13:28:30   Log-Likelihood:                -984.55
converged:                       True   LL-Null:                       -984.59
Covariance Type:            nonrobust   LLR p-value:                    0.7942
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.4478      0.053      8.410      0.000       0.343       0.552
income_10k     0.0012      0.

Interpret the coefficient on `income_10k`. Do not read it as you would an OLS coefficient.

Poisson coefficients are **factors of change**: a one-unit change in `income_10k` is expected to change the number of children by a factor of exp(β), ceteris paribus.

You can alternatively read this as a **percentage change**: 100 × [exp(β) − 1]%.

In [5]:
import numpy as np
b = po_lm1.params['income_10k']
print(f'Factor of change, exp(b): {np.exp(b):.4f}')
print(f'Percentage change, 100 * [exp(b) - 1]: {100 * (np.exp(b) - 1):.2f}%')

Factor of change, exp(b): 1.0012
Percentage change, 100 * [exp(b) - 1]: 0.12%


# Adding Age

Add `age` to both models. `age` is centered at 18, so 0 is an 18-year-old. Save as `ols_lm2` and `po_lm2`.

In [6]:
ols_lm2 = smf.ols('biokids ~ income_10k + age', data=df).fit()
po_lm2  = smf.poisson('biokids ~ income_10k + age', data=df).fit()
print(po_lm2.summary())

Optimization terminated successfully.
         Current function value: 1.720780
         Iterations 5
                          Poisson Regression Results                          
Dep. Variable:                biokids   No. Observations:                  557
Model:                        Poisson   Df Residuals:                      554
Method:                           MLE   Df Model:                            2
Date:                Fri, 16 Oct 2026   Pseudo R-squ.:                 0.02652
Time:                        13:28:36   Log-Likelihood:                -958.47
converged:                       True   LL-Null:                       -984.59
Covariance Type:            nonrobust   LLR p-value:                 4.563e-12
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0674      0.094     -0.720      0.472      -0.251       0.116
income_10k     0.0008      0.

## Comparing the Poisson Models via Likelihood Ratio Test

In [7]:
lr_stat = 2 * (po_lm2.llf - po_lm1.llf)
df_diff = po_lm2.df_model - po_lm1.df_model
p_val   = chi2.sf(lr_stat, df_diff)
print(f'LR statistic = {lr_stat:.4f}, df = {df_diff:.0f}, p-value = {p_val:.6f}')

LR statistic = 52.1579, df = 1, p-value = 0.000000


# Building a Negative Binomial Model

Negative binomial relaxes the Poisson assumption that variance = mean (allowing overdispersion).

Equivalent to R's `MASS::glm.nb()`.

Read the coefficients the same way as in a Poisson model: as a factor of change, exp(β), or as a percentage change, 100 × [exp(β) − 1]%.

In [8]:
nb_m1 = smf.negativebinomial('biokids ~ income_10k + age', data=df).fit()
print(nb_m1.summary())

Optimization terminated successfully.
         Current function value: 1.670251
         Iterations: 14
         Function evaluations: 19
         Gradient evaluations: 19
                     NegativeBinomial Regression Results                      
Dep. Variable:                biokids   No. Observations:                  557
Model:               NegativeBinomial   Df Residuals:                      554
Method:                           MLE   Df Model:                            2
Date:                Fri, 16 Oct 2026   Pseudo R-squ.:                 0.01718
Time:                        13:28:41   Log-Likelihood:                -930.33
converged:                       True   LL-Null:                       -946.59
Covariance Type:            nonrobust   LLR p-value:                 8.681e-08
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0832      0.116     -

In [9]:
from stargazer.stargazer import Stargazer

sg = Stargazer([ols_lm2, po_lm2, nb_m1])
sg.title('Number of Children')
sg.custom_columns(['OLS', 'Poisson', 'Neg. Binomial'], [1, 1, 1])

# Ending the cell on the object itself renders the table in the notebook
sg

## Test for Overdispersion

Was NB necessary? We test whether the NB's dispersion parameter (alpha) is significantly different from 0. If alpha = 0, the NB reduces to Poisson.

This is equivalent to R's `pscl::odTest()`.

In [10]:
# Overdispersion test: LR comparing NB vs Poisson
# Under H0 (Poisson), the test statistic has a chi-square(1)/2 distribution
# because the null is on the boundary of the parameter space
lr_od = 2 * (nb_m1.llf - po_lm2.llf)
p_od  = 0.5 * chi2.sf(lr_od, 1)
print(f'Overdispersion LR statistic = {lr_od:.4f}')
print(f'p-value (one-sided) = {p_od:.6f}')
print('NB preferred over Poisson:', p_od < 0.05)

Overdispersion LR statistic = 56.2895
p-value (one-sided) = 0.000000
NB preferred over Poisson: True


# Marginal Effects

A marginal effect is the expected change in the number of children when an independent variable increases by one unit, ceteris paribus. Compute them the same way as for the logit models.

In [11]:
from marginaleffects import avg_comparisons, comparisons

## Average marginal effects of income

In [12]:
avg_comparisons(po_lm2, variables='income_10k')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""income_10k""","""+1""",0.001238,0.007496,0.165144,0.868891,0.202754,-0.013487,0.015963


In [13]:
avg_comparisons(nb_m1, variables='income_10k')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""income_10k""","""+1""",0.00077,0.009532,0.080801,0.93563,0.095991,-0.017952,0.019493


## Marginal effect at the mean of income

In [14]:
comparisons(po_lm2, variables='income_10k', newdata='mean')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""income_10k""","""+1""",0.001201,0.007274,0.165154,0.868883,0.202767,-0.013086,0.015488


In [15]:
comparisons(nb_m1, variables='income_10k', newdata='mean')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""income_10k""","""+1""",0.000746,0.009229,0.080801,0.935629,0.095991,-0.017383,0.018874


# Independent Investigation: Education and Number of Children

Investigate the relationship between **no. of children** (`biokids`) and **BA education** (`ba_binary`).

Do the following:

1. Include these independent variables: `ba_binary`, `age`, `income_10k`, `sex_woman`, and `race_bin`.
2. Build OLS, Poisson, and Negative Binomial models.
3. Produce a Stargazer table comparing the three models.
4. Compute marginal effects for each model.
5. Decide which model is best. Justify.
6. Answer: does a person with a BA have more or fewer children, and by how much? For the Poisson and negative binomial models, report a factor of change or a percentage change.

In [16]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

In [21]:
print(df.columns.tolist())

['id', 'age', 'happy', 'yrlvmus', 'yrartxbt', 'yrmovie', 'yrcreat', 'income16', 'prestg10', 'degree', 'race', 'sex', 'wrkstat', 'biokids', 'educ', 'income_cont', 'income_10k', 'binary_lvmus', 'binary_artxbt', 'binary_movie', 'binary_creat', 'omni', 'race_bin', 'race_cat', 'ba_binary', 'sex_woman', 'working', 'happy_bin', 'happy_cat']


In [26]:
ccols = [
    "biokids", "ba_binary", "age",
    "income_10k", "sex_woman", "race_bin"
]

df = df[cols].dropna().copy()

print(df.head())
print(df.shape)

    biokids  ba_binary   age  income_10k  sex_woman race_bin
4       2.0          0  44.0       2.750  Not Woman      1.0
17      7.0          1  62.0      14.000      Woman      1.0
18      4.0          0  17.0       1.125      Woman      2.0
20      3.0          0  16.0       4.500      Woman      3.0
22      0.0          1  47.0       3.750  Not Woman      1.0
(557, 6)


In [27]:
formula = "biokids ~ ba_binary + age + income_10k + sex_woman + race_bin"

ols = smf.ols(formula, data=df_diff).fit()
poisson = smf.poisson(formula, data=df).fit()
nb = smf.negativebinomial(formula, data=df).fit()

print(ols.summary())
print(poisson.summary())
print(nb.summary())

Optimization terminated successfully.
         Current function value: 1.671346
         Iterations 5
Optimization terminated successfully.
         Current function value: 1.637030
         Iterations: 28
         Function evaluations: 34
         Gradient evaluations: 34
                            OLS Regression Results                            
Dep. Variable:                biokids   R-squared:                       0.118
Model:                            OLS   Adj. R-squared:                  0.109
Method:                 Least Squares   F-statistic:                     12.29
Date:                Fri, 16 Oct 2026   Prob (F-statistic):           5.36e-13
Time:                        13:41:30   Log-Likelihood:                -1015.5
No. Observations:                 557   AIC:                             2045.
Df Residuals:                     550   BIC:                             2075.
Df Model:                           6                                         
Covariance Type

In [28]:
from stargazer.stargazer import Stargazer
from IPython.display import HTML, display

stargazer = Stargazer([ols, poisson, nb])

stargazer.title("Education and Number of Children")
stargazer.custom_columns(
    ["OLS", "Poisson", "Negative Binomial"],
    [1, 1, 1]
)

display(HTML(stargazer.render_html()))

In [29]:
#Marginal Effects
#OLS
print("OLS Marginal Effects:")
print(ols.params)

OLS Marginal Effects:
Intercept             0.483667
sex_woman[T.Woman]    0.493528
race_bin[T.2.0]       0.626281
race_bin[T.3.0]       0.064282
ba_binary            -0.481404
age                   0.023289
income_10k            0.021642
dtype: float64


In [30]:
# Poisson
print("Poisson Marginal Effects:")
print(poisson.get_margeff(at="overall", dummy=True).summary())

Poisson Marginal Effects:
       Poisson Marginal Effects      
Dep. Variable:                biokids
Method:                          dydx
At:                           overall
                        dy/dx    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
sex_woman[T.Woman]     0.4958      0.107      4.614      0.000       0.285       0.706
race_bin[T.2.0]        0.6746      0.195      3.454      0.001       0.292       1.057
race_bin[T.3.0]        0.0628      0.257      0.245      0.807      -0.440       0.566
ba_binary             -0.4857      0.117     -4.169      0.000      -0.714      -0.257
age                    0.0244      0.003      7.278      0.000       0.018       0.031
income_10k             0.0224      0.008      2.738      0.006       0.006       0.038


In [31]:
# Negative Binomial
print("Negative Binomial Marginal Effects:")
print(nb.get_margeff(at="overall", dummy=True).summary())

Negative Binomial Marginal Effects:
  NegativeBinomial Marginal Effects  
Dep. Variable:                biokids
Method:                          dydx
At:                           overall
                        dy/dx    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
sex_woman[T.Woman]     0.5059      0.133      3.816      0.000       0.246       0.766
race_bin[T.2.0]        0.6854      0.251      2.728      0.006       0.193       1.178
race_bin[T.3.0]        0.0668      0.312      0.214      0.830      -0.544       0.678
ba_binary             -0.5154      0.145     -3.549      0.000      -0.800      -0.231
age                    0.0254      0.004      6.002      0.000       0.017       0.034
income_10k             0.0237      0.010      2.303      0.021       0.004       0.044


Holding age, income, gender, and race constant, having a BA degree is associated with approximately 0.481 fewer children in the OLS model, 0.486 fewer children in the Poisson model, and 0.515 fewer children in the Negative Binomial model.statistically significant suggest a negative association between higher education and the number of children.

In [33]:
#AIC
print("OLS AIC:", ols.aic)
print("Poisson AIC:", poisson.aic)
print("Negative Binomial AIC:", nb.aic)

OLS AIC: 2044.9385308345434
Poisson AIC: 1875.8797265793137
Negative Binomial AIC: 1839.6515844996736


Negative Binomial model is the most appropriate among the three models.
Negative Binomial model has the lowest AIC (1839.65)